![PIC UPV PERTE Chip Chair Logo](https://www.pic-chair.upv.es/wp-content/uploads/2024/05/logo-upv-horizontal.png)
![PIC UPV PERTE Chip Chair Logo](https://www.pic-chair.upv.es/wp-content/uploads/2024/06/logos-perte-chip-1024x119.png)


# Laboratory 0.3. Ring Resonators

**Duration:** 2 hours | **Tools:** Python, gplugins/Tidy3D mode solver, NumPy, Matplotlib  
**Deliverable:** Fill in the companion `LabBook_030.pptx` with figures, numerical observations and short answers.

We will reuse the SiN/SiO₂ waveguide cross-section from **Lab 0.1**, then use a **simplified analytical circuit model** for an all-pass ring resonator. This is **not** a full electromagnetic simulation of the bent ring or its curved coupling region.

**Learning outcomes**
1. Connect the effective index of a waveguide to ring resonance.
2. Analyze how the ring radius affects the spectral response and FSR.
3. Explain how coupling and round-trip loss affect the depth of transmission dips.

## 0. Imports — run first
Run the cells in order, as in Labs 0.1 and 0.2. Keep dimensions in **µm** inside the Python calculations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gplugins.tidy3d as gt

# Units
nm = 1e-3  # 1 nm expressed in µm
# Reference wavelength
lambda0 = 1.55  # µm
# Wavelength sweep for modal analysis
wavelength_modes = np.linspace(1.50, 1.60, 11)  # µm

# Wavelength sweep for ring spectral analysis
lambda_sweep = np.linspace(1.50, 1.60, 4001)  # µm

## LO.1. Effective and Group Index Analysis
### 1.0. Reuse the mode solver from Lab 0.1 

The cross-section is the same: **1.2 µm wide, 300 nm high SiN core in SiO₂**. The **fundamental TE-like mode** is used for this introductory example. Inspect the TE fractions if the ordering is different in your solver version.

In [ ]:
# Waveguide geometry (same as Lab 0.1)

deep_waveguide = gt.modes.Waveguide(
    core_width=1.2,
    core_thickness=300 * nm,
    slab_thickness=0 * nm,
    core_material='sin',
    clad_material='sio2',
    wavelength=wavelength_modes,
    num_modes=4,
    max_grid_scaling=1.5,
    grid_resolution=20,
    cache_path='.cache/',
    precision='double',
)

# Calculate effective indices and TE fractions
print('Effective indices:', deep_waveguide.n_eff)
print('TE fractions:', deep_waveguide.fraction_te)


### 1.1. Extract the effective index of the fundamental TE mode for different wavelengths

In [ ]:
# Select fundamental TE-like mode
te_mode_index = 0

# Extract effective index
neff = np.real(
    np.asarray(deep_waveguide.n_eff)[:, te_mode_index]
)

# Plot effective index vs wavelength
plt.figure(figsize=(8, 4))
plt.plot(wavelength_modes * 1000, neff, 'o-')

plt.xlabel('Wavelength (nm)')
plt.ylabel('Effective index')
plt.title('Effective index vs wavelength')
plt.grid(True)
plt.show()

# Effective index at 1550 nm
idx = np.argmin(np.abs(wavelength_modes - lambda0))
neff_1550 = neff[idx]

print(f'n_eff at 1550 nm = {neff_1550:.5f}')

Note: The effective index, n_eff(wvl), is expected to vary smoothly with wavelength. If the plot shows unexpected discontinuities or abrupt changes, the selected mode may not correspond to the fundamental TE-like mode across the entire wavelength range.

In this case, check the effective indices and TE fractions to correctly identify the fundamental TE-like mode, and adjust te_mode_index if necessary.

Important: The effective index must always be obtained from the modal simulation, not replaced by an arbitrary value

### 1.2. Calculate and plot the group index

In [ ]:
# Numerical derivative of effective index
dneff_dlambda = np.gradient(neff, wavelength_modes)

# Calculate group index
ng = neff - wavelength_modes * dneff_dlambda

# Plot group index vs wavelength
plt.figure(figsize=(8, 4))
plt.plot(wavelength_modes * 1000, ng, 'o-')

plt.xlabel('Wavelength (nm)')
plt.ylabel('Group index')
plt.title('Group index vs wavelength')
plt.grid(True)
plt.show()

# Group index at 1550 nm
ng_1550 = ng[idx]

print(f'n_g at 1550 nm = {ng_1550:.5f}')

In [ ]:
te_mode_index = 0  # Check the TE fractions and change only if necessary
neff_1550 = float(np.real(np.asarray(deep_waveguide.n_eff).reshape(-1)[te_mode_index]))
print(f'Fundamental TE-like n_eff(1550 nm) = {neff_1550:.5f}')


### LO.2. Ring Resonator Spectral Response 

A ring resonator consists of a closed-loop waveguide coupled to a straight waveguide. At specific wavelengths, light circulating inside the ring satisfies the resonance condition:

$$
m\lambda = n_{\mathrm{eff}}(\lambda)L
$$

where $L = 2\pi R$ is the ring circumference, $R$ is the ring radius, and $m$ is an integer representing the resonance order.

The round-trip phase accumulated by light propagating around the ring is:

$$
\phi(\lambda) = \frac{2\pi n_{\mathrm{eff}}(\lambda)L}{\lambda}
$$

According to the lecture, the through-port power transmission is given by:

$$
T_p(\lambda) = \frac{1+x^2y^2-2xy\cos\phi(\lambda)}{x^2+y^2-2xy\cos\phi(\lambda)}
$$

where:

- $x = 1/a$ is the loss parameter, with $a$ representing the round-trip field-amplitude transmission.
- $y$ is the self-coupling field coefficient between the straight waveguide and the ring.
- $\phi(\lambda)$ is the round-trip phase accumulated inside the ring.

In this exercise, we will use the wavelength-dependent effective index $n_{\mathrm{eff}}(\lambda)$ calculated in LO.1 to simulate the spectral response of an all-pass ring resonator.


### 2.2. Interpolate the Effective Index

In LO.1, we calculated the effective index of the fundamental TE mode at 11 wavelengths between 1500 and 1600 nm.

To accurately represent the narrow resonances of a ring resonator, we need a finer wavelength resolution.

Instead of running additional mode simulations, we will use **linear interpolation** to estimate $n_{\mathrm{eff}}(\lambda)$ over 4001 wavelength points.

This interpolated effective index will be used to calculate the round-trip phase and the spectral response of the ring resonator.

In [ ]:
# Interpolate effective index over the wavelength range

neff_sweep = np.interp(
    lambda_sweep,
    wavelength_modes,
    neff
)

# Plot original and interpolated effective index
plt.figure(figsize=(8, 4))

plt.plot(lambda_sweep * 1000, neff_sweep, '-',
         label='Interpolated effective index')

plt.plot(wavelength_modes * 1000, neff, 'o',
         label='Mode solver results')

plt.xlabel('Wavelength (nm)')
plt.ylabel('Effective index')
plt.title('Effective Index Interpolation')
plt.legend()
plt.grid(True)
plt.show()

#### QUESTION: Why do we interpolate the effective index instead of running the mode solver at every wavelength point?


### 2.3. Ring Resonator Transmission Spectrum

We will now calculate the through-port transmission spectrum of an all-pass ring resonator using the wavelength-dependent effective index obtained in Section 2.2.

The ring parameters are:

- **Ring radius:** $R = 10$ µm
- **Round-trip field-amplitude transmission:** $a = 0.90$
- **Self-coupling field coefficient:** $y = 0.90$
- **Wavelength range:** 1500–1600 nm

The round-trip phase is calculated as:

$$
\phi(\lambda) = \frac{2\pi n_{\mathrm{eff}}(\lambda) \cdot 2\pi R}{\lambda}
$$

The through-port power transmission is:

$$
T_p(\lambda) = \frac{1+x^2y^2-2xy\cos\phi(\lambda)}{x^2+y^2-2xy\cos\phi(\lambda)}
$$

where $x=1/a$.

We will use these expressions to calculate and plot the transmission spectrum of the resonator.

In [ ]:
# Ring resonator transmission function

def ring_transmission(wavelength_um, radius_um, neff, a=0.90, y=0.90):
    """Calculate the through-port power transmission of an all-pass ring."""

    # Ring circumference (µm)
    L = 2 * np.pi * radius_um

    # Round-trip phase
    phi = 2 * np.pi * neff * L / wavelength_um

    # Loss parameter
    x = 1 / a

    # Power transmission
    numerator = 1 + (x * y)**2 - 2 * x * y * np.cos(phi)
    denominator = x**2 + y**2 - 2 * x * y * np.cos(phi)

    return numerator / denominator


# Ring parameters
radius = 10.0  # µm
a = 0.90       # Round-trip amplitude transmission
y = 0.90       # Self-coupling amplitude coefficient

# Calculate transmission
transmission = ring_transmission(
    lambda_sweep,
    radius,
    neff_sweep,
    a,
    y
)

# Plot transmission spectrum
plt.figure(figsize=(9, 4))

plt.plot(lambda_sweep * 1000, transmission)

plt.xlabel('Wavelength (nm)')
plt.ylabel('Through-port power transmission')
plt.title('Ring Resonator Transmission Spectrum (R = 10 µm)')
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.show()


#### Key Observations

1) Resonance Depth: The through-port transmission drops to nearly zero at the resonant wavelengths due to the critical coupling condition ($a = y = 0.90$).

2) Free Spectral Range (FSR): The spacing between consecutive resonances increases slightly with wavelength (approximately 19–21 nm), consistent with the wavelength dependence of the FSR.

3) Off-Resonance Transmission: Between resonances, the transmission approaches unity, indicating that most of the optical power continues propagating through the straight waveguide rather than being coupled into the ring.

### LO.3. Ring Resonator Design Parameters and Spectral Performance
### 3.1. Influence of Coupling and Losses

The spectral response of a ring resonator depends on two main parameters:

- **Round-trip amplitude transmission ($a$):** represents the optical losses inside the ring.
- **Self-coupling amplitude coefficient ($y$):** determines how light is coupled between the straight waveguide and the ring.

Depending on the relationship between these parameters, three coupling regimes can be identified:

- **Undercoupled:** $y > a$
- **Critically coupled:** $y = a$
- **Overcoupled:** $y < a$

#### Exercise 1 – Influence of the coupling coefficient

Using the `ring_transmission()` function developed in Section 2.3, investigate how the coupling coefficient affects the spectral response.

Keep the ring radius and round-trip losses constant:

- Ring radius: $R = 10$ µm
- Round-trip amplitude transmission: $a = 0.90$

Simulate the following cases:

| Case | Self-coupling coefficient ($y$) | Coupling regime |
|---|---|---|
| A | 0.99 | Undercoupled |
| B | 0.90 | Critically coupled |
| C | 0.70 | Overcoupled |

**Tasks:**

1. Plot the three transmission spectra on the same graph.
2. Compare the resonance depths for the three coupling regimes.
3. Identify which configuration produces the deepest transmission minima.
4. Explain why the through-port transmission approaches zero under critical coupling.

#### Exercise 2 – Influence of optical losses 

Keep the ring radius and coupling coefficient constant:

- Ring radius: $R = 10$ µm
- Self-coupling coefficient: $y = 0.90$

Simulate three different values of round-trip amplitude transmission:

| Case | Round-trip amplitude transmission ($a$) |
|---|---|
| D | 0.99 |
| E | 0.90 |
| F | 0.75 |

**Questions:**

1. How does increasing the propagation loss affect the resonance depth?
2. Which configuration satisfies the critical coupling condition?
3. Can increasing the internal losses produce a deeper resonance? Explain why.

### 3.2. Influence of Ring Radius on the Free Spectral Range (25 min)

The **Free Spectral Range (FSR)** is the spectral separation between two consecutive resonances of a ring resonator.

The FSR expressed in wavelength can be approximated as:

$$
FSR_\lambda \approx \frac{\lambda^2}{n_g(\lambda)L}
$$

where:

- $\lambda$ is the resonance wavelength.
- $n_g$ is the group index calculated in LO.1.
- $L = 2\pi R$ is the ring circumference.

This expression shows that the FSR depends on both the group index and the physical length of the resonator.

#### Exercise 3 – Influence of Ring Radius

Using the `ring_transmission()` function developed in Section 2.3, investigate how the ring radius affects the spectral response.

Keep the coupling and loss parameters constant:

- Round-trip amplitude transmission: $a = 0.90$
- Self-coupling coefficient: $y = 0.90$
- Wavelength range: 1500–1600 nm

Simulate the following ring radii:

| Case | Ring radius ($R$) |
|---|---|
| A | 5 µm |
| B | 10 µm |
| C | 20 µm |

**Tasks:**

1. Plot the transmission spectra of the three resonators and compare their resonance spacing.
2. Identify two consecutive resonances near 1550 nm and calculate the FSR for each ring radius.
3. Using the group index obtained in LO.1, calculate the theoretical FSR at 1550 nm for each radius.
4. Complete the following table and compare the numerical and theoretical results.

| Radius (µm) | Simulated FSR (nm) | Theoretical FSR (nm) |
|-----|----|----|
| 5 | | |
| 10 | | |
| 20 | | |

**Questions:**

1. How does the FSR change when the ring radius increases?
2. Explain physically why the FSR depends on the ring circumference.
3. Why is the FSR not exactly constant across the wavelength range?

**Note:** In this exercise, the effective and group indices are assumed to be independent of the ring radius. Bending-induced changes in the guided mode and propagation losses are neglected.

In [ ]:

# Exercise 3 - Influence of ring radius on FSR

# Fixed parameters
a = 0.90
y = 0.90

# Ring radii (µm)
radii = [5, 10, 20]

# Plot transmission spectra
plt.figure(figsize=(10, 5))

for R in radii:
    T = ring_transmission(
        lambda_sweep,
        R,
        neff_sweep,
        a,
        y
    )

    plt.plot(
        lambda_sweep * 1000,
        T,
        label=f'R = {R} µm'
    )

plt.xlabel('Wavelength (nm)')
plt.ylabel('Through-port power transmission')
plt.title('Influence of Ring Radius on FSR')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Theoretical FSR at 1550 nm

for R in radii:
    L = 2 * np.pi * R

    FSR_um = lambda0**2 / (ng_1550 * L)
    FSR_nm = FSR_um * 1000

    print(f'R = {R:2d} µm -> Theoretical FSR = {FSR_nm:.2f} nm')


### 3.3. Resonance Linewidth and Applications 

The spectral performance of a ring resonator is not only determined by its Free Spectral Range (FSR), but also by the **width of its resonances**.

The **Full Width at Half Maximum (FWHM)** describes the spectral width of a resonance. For a transmission dip, it is measured at half of the resonance depth, relative to the off-resonance transmission level.

Two additional parameters are commonly used to characterize optical resonators:

**Finesse:**

$$
\mathcal{F} = \frac{FSR}{FWHM}
$$

**Loaded Quality Factor (Q-factor):**

$$
Q = \frac{\lambda_R}{FWHM}
$$

where $\lambda_R$ is the resonance wavelength and the FWHM is expressed in the same wavelength units.

A narrower resonance generally corresponds to higher finesse and a higher loaded Q-factor, for a fixed resonance wavelength and FSR.

#### Exercise 4 – Influence of Coupling and Losses on Resonance Linewidth

Consider two critically coupled ring resonators with the same radius but different round-trip losses and coupling coefficients.

| Parameter | Ring A | Ring B |
|---|---|---|
| Radius ($R$) | 10 µm | 10 µm |
| Round-trip amplitude transmission ($a$) | 0.90 | 0.99 |
| Self-coupling coefficient ($y$) | 0.90 | 0.99 |
| Coupling regime | Critical | Critical |

Using `ring_transmission()`, simulate both resonators and compare their transmission spectra.

**Student tasks:**

1. Plot both transmission spectra over the wavelength range of 1500–1600 nm.
2. Zoom in on a resonance near 1550 nm and compare the resonance linewidths.
3. Graphically estimate the FWHM of each resonance by identifying the two wavelengths at half of the resonance depth (approximately T = 0.5). No additional simulations are required.
4. Calculate the finesse and loaded Q-factor for both resonators.

Complete the following table:

| Parameter | Ring A | Ring B |
|---|---|---|
| Resonance wavelength (nm) | | |
| FSR (nm) | | |
| Estimated FWHM (nm) | | |
| Finesse | | |
| Loaded Q-factor | | |

#### Discussion – Applications

Ring resonators are widely used in integrated photonics for optical filtering, sensing and wavelength-selective devices.

**Questions:**

1. Which resonator has the narrower resonance and higher Q-factor?
2. Why does reducing the round-trip loss, while maintaining critical coupling, produce a narrower resonance?
3. Which resonator would be more suitable for detecting small resonance wavelength shifts in an optical sensor? Explain your choice.
4. Is a narrower resonance always advantageous for wavelength-division multiplexing (WDM) filtering? Consider the trade-off between spectral selectivity and operational bandwidth.


In [ ]:

# Exercise 4 - Resonance linewidth comparison

radius = 10.0  # µm

# Ring A: higher losses
a_A = 0.90
y_A = 0.90

# Ring B: lower losses
a_B = 0.99
y_B = 0.99

# Calculate transmission spectra
T_A = ring_transmission(
    lambda_sweep, radius, neff_sweep, a_A, y_A
)

T_B = ring_transmission(
    lambda_sweep, radius, neff_sweep, a_B, y_B
)

# Plot full spectra
plt.figure(figsize=(10, 4))

plt.plot(lambda_sweep * 1000, T_A,
         label='Ring A: a = y = 0.90')

plt.plot(lambda_sweep * 1000, T_B,
         label='Ring B: a = y = 0.99')

plt.xlabel('Wavelength (nm)')
plt.ylabel('Through-port power transmission')
plt.title('Comparison of Resonance Linewidths')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from scipy.signal import find_peaks


# Zoom around a resonance near 1550 nm

plt.figure(figsize=(10, 5))

plt.plot(lambda_sweep * 1000, T_A,
         label='Ring A: a = y = 0.90')

plt.plot(lambda_sweep * 1000, T_B,
         label='Ring B: a = y = 0.99')

# Zoom around the resonance
plt.xlim(1547, 1555)
plt.ylim(-0.05, 1.05)

plt.xlabel('Wavelength (nm)')
plt.ylabel('Through-port power transmission')
plt.title('Resonance Linewidth Comparison')
plt.legend()
plt.grid(True)
plt.show()